<a href="https://colab.research.google.com/github/202501090032/my-ct-project-1/blob/main/Prompt_Engineering_HuggingFace_ipynbb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install --upgrade pip -q
!pip install -U transformers accelerate huggingface_hub torch sentencepiece -q

In [1]:
from huggingface_hub import login
import getpass

print("Enter Hugging Face token (or press Enter to skip):")

token = getpass.getpass()

if token.strip() != "":
    login(token=token)
    print("Logged in successfully!")
else:
    print("Skipping login...")

Enter Hugging Face token (or press Enter to skip):
··········
Logged in successfully!


In [3]:
history = []

print("AI Chatbot Started!")
print("Type 'quit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Chat ended.")
        break

    history.append(f"User: {user_input}")

    prompt = "\n".join(history[-6:]) + "\nAssistant:"

    response = chatbot(
        prompt,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True
    )

    reply = response[0]["generated_text"].split("Assistant:")[-1].strip()

    print("Bot:", reply)

    history.append(f"Assistant: {reply}")

AI Chatbot Started!
Type 'quit' to stop.

You: hello


NameError: name 'chatbot' is not defined

In [12]:
from transformers import pipeline
import gradio as gr

chatbot = pipeline(
    "text-generation",
    model="distilgpt2"
)

def chat(message, history):
    # Keep short memory for speed + stability
    history = history[-5:]

    # Build clean prompt style (IMPORTANT for less randomness)
    prompt_prefix = "The following is a friendly conversation with an AI assistant.\n\n"
    current_conversation_turns = ""

    for user_msg, bot_msg in history:
        current_conversation_turns += f"User: {user_msg}\nAI: {bot_msg}\n"

    # Construct the full prompt that will be sent to the model
    full_prompt_for_generation = prompt_prefix + current_conversation_turns + f"User: {message}\nAI:"

    result = chatbot(
        full_prompt_for_generation,
        max_new_tokens=60,
        temperature=0.6,
        do_sample=True,
        return_full_text=True # Ensure we get the full text including the prompt
    )

    generated_text = result[0]["generated_text"]

    # Extract only the part generated by the AI after our prompt
    # We expect the generated_text to start with full_prompt_for_generation
    if generated_text.startswith(full_prompt_for_generation):
        raw_reply = generated_text[len(full_prompt_for_generation):].strip()
    else:
        # Fallback if the generated text structure is unexpected
        raw_reply = generated_text.strip()

    # Further clean the raw_reply to remove any potential subsequent 'User:' or 'AI:' tags
    # taking only the first coherent AI utterance.
    final_reply = raw_reply
    if "User:" in raw_reply:
        final_reply = raw_reply.split("User:")[0].strip()
    elif "AI:" in raw_reply: # This case is less likely if split by User: first
        final_reply = raw_reply.split("AI:")[0].strip()

    # Clean fallback if model goes weird or generates the same input
    if len(final_reply) == 0 or final_reply.lower().strip() == message.lower().strip():
        final_reply = "I'm here to help you 😊"
    # Additional check: if the reply starts with the user's message, it might be a partial repetition.
    elif final_reply.lower().startswith(message.lower()):
        # Try to extract something meaningful after the repetition if it's long enough
        if len(final_reply) > len(message) + 5: # If there's significantly more text
            final_reply = final_reply[len(message):].strip()
            # Remove any leading 'AI:' or 'User:' tags that might have been part of the repetition
            if final_reply.lower().startswith("ai:"):
                final_reply = final_reply[3:].strip()
            if final_reply.lower().startswith("user:"):
                final_reply = final_reply[5:].strip()
            if len(final_reply) == 0:
                final_reply = "I'm here to help you 😊"
        else:
            final_reply = "I'm here to help you 😊"

    return final_reply

demo = gr.ChatInterface(
    fn=chat,
    title="⚡ FAST AI CHATBOT (NO LAG)",
    description="Optimized lightweight chatbot for Colab"
)

demo.launch()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08308e459d8efc22ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
